# Winiwarter2021 scene loading time

Run the two cells below using a Python kernel with `heliospp` installed. The timer measures XML validation and scene loading, excluding imports and path preparation. No simulation or visualization is run. Restart the kernel before repeating to release the previous scene; filesystem caching can affect repeated timings.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
import xml.etree.ElementTree as ET

import helios

repo = Path("/home/thomas/repositories/helios")
dataset = repo / "data_Winiwarter2021"
scene_xml = dataset / "scenes/wellington_citrusorchard_architecture/scene.xml"
helios.add_asset_directory(repo)

# Correct legacy Windows paths and the dataset's renamed data directory.
# Keep the original XML unchanged; preparation is outside the timer.
tree = ET.parse(scene_xml)
for param in tree.iter("param"):
    if param.get("key") == "filepath":
        relative = param.get("value").replace("\\", "/")
        asset = dataset / relative.removeprefix("data/")
        if not asset.is_file():
            asset = repo / relative
        if not asset.is_file():
            raise FileNotFoundError(asset)
        param.set("value", str(asset))

temporary_directory = TemporaryDirectory(prefix="helios-scene-timing-")
prepared_xml = Path(temporary_directory.name) / "scene.xml"
tree.write(prepared_xml, encoding="utf-8", xml_declaration=True)

In [ ]:
start = perf_counter()
scene = helios.StaticScene.from_xml(prepared_xml)
elapsed = perf_counter() - start
print(f"Scene loaded in {elapsed:.3f} seconds ({elapsed / 60:.2f} minutes).")